# ADAC Workshop 1 26L: Performance & Computing Models

## Introduction
In this lab, you will compare the performance and computing models of four popular data processing libraries/engines: **Polars, Pandas and DuckDB**. We will include PySpark distributed engine in the introduction, but we will not focus on distributed execution in this lab.

You will explore:
- **Performance**: single-node processing speed, parallel execution, memory usage, and result materialization cost.
- **Scalability**: how performance changes with the number of local threads/cores.
- **Physical layout**: how file format, Parquet layout, row groups, sorting, partitioning, and pruning affect IO.
- **Computing models**: in-memory vs. out-of-core processing, SQL vs. DataFrame APIs, eager vs. lazy execution, and streaming execution vs. streaming output.


## Submission identity

Before starting the assignment, copy this notebook into your fork of the workshop repository and work on that copy.

Fill in the first code cell with:

- your full name,
- a link to this notebook in your forked GitHub repository,
- names or IDs of group members if required by the instructor.

The submitted notebook should be reachable from your fork. Do not submit a notebook that only exists locally.
The notebook is stored in the repository: https://github.com/bdg-tbd/tbd-workshop-1/tree/ADAC26L/notebooks/ . TBD is another WUT course that uses similar notebooks to ADAC, so we use the same repository for both courses. 

In [ ]:
# TODO: Fill this in before submitting. Change <your-github-user-or-org> and <branch>
FULL_NAME = None
NOTEBOOK_URL = "https://github.com/<your-github-user-or-org>/tbd-workshop-1/blob/<branch>/notebooks/adac_workshop_1.ipynb"

assert FULL_NAME is not None, "Set FULL_NAME before running the notebook"
assert "<your-github-user-or-org>" not in NOTEBOOK_URL, "Set NOTEBOOK_URL to your forked repository notebook URL"
assert "<branch>" not in NOTEBOOK_URL, "Set NOTEBOOK_URL to your forked repository notebook URL"

## Library/engine capabilities

Use this table as a reference when interpreting your results.

| Library/engine | Query optimizer | Distributed | Arrow-backed | Out-of-core | Parallel local execution | Main APIs |
|---|---|---|---|---|---|---|
| **Pandas 3.0.2** | no | no | default IO returns NumPy-backed data; `dtype_backend="pyarrow"` returns PyArrow-backed nullable dtypes | no | limited | DataFrame, `pd.col` for selected expression-style usage |
| **Polars** | yes | single-node locally; distributed engine is available in Polars Cloud and is outside this local benchmark | yes | yes | yes | DataFrame, lazy expressions, SQL subset |
| **DuckDB** | yes | no | no, but Arrow-compatible (zero-copy interop) | yes | yes | SQL, relational API |
| **PySpark** | yes | yes | no | yes | yes | SQL, DataFrame |

The goal is not to prove that one library is always best. The goal is to identify which library/engine is appropriate for a given data size, query shape, memory limit, physical layout, and deployment model.

Use pandas 3.0.2 in this lab. Two pandas 3.0.2 behaviours matter for the benchmark:
- string columns are no longer inferred as generic `object` dtype by default,
- Copy-on-Write is the only mutation model (in pandas 2.x there could be unintended side effects of mutating DataFrames - in 3.x every object behaves as an independent copy).

In addition, compare two Pandas Parquet-reading variants where possible:  
- default Pandas/NumPy-backed DataFrame: `pd.read_parquet(path)`,
- PyArrow-backed DataFrame: `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`.

Record the pandas version and dtypes in your report.


## Parquet introduction

Image and introduction reference: [Parquet pruning pipeline in DataFusion](https://datafusion.apache.org/blog/2025/03/20/parquet-pruning/)

![Parquet pruning pipeline in DataFusion](https://datafusion.apache.org/blog/images/parquet-pruning/read-parquet.jpg)

## Prerequisites

Install the required libraries in your notebook environment. Pandas 3.0.2 requires Python 3.11 or newer.

Use current Polars API in new code. In particular, use `collect(engine="streaming")` for streaming execution and use sink methods when you want to write streaming output to disk.

For Pandas, benchmark both the default backend and the PyArrow dtype backend for Parquet reads. The PyArrow-backed variant is especially relevant for string-heavy datasets.

Use the cell below to install the dependencies:


In [ ]:
%pip install -U "pandas>=3.0,<3.1" polars duckdb memory_profiler psutil matplotlib seaborn

In [ ]:
import gc
import os
import time
import json
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import duckdb
import psutil
from memory_profiler import memory_usage
import pyarrow.parquet as pq
import pyarrow

print("Python:", platform.python_version())
if tuple(map(int, platform.python_version_tuple()[:2])) < (3, 11):
    raise RuntimeError("This notebook requires Python 3.11+ because it uses pandas 3.0.2.")
print("Polars:", pl.__version__)
print("Pandas:", pd.__version__)
print("PyArrow:", pyarrow.__version__)
if tuple(map(int, pd.__version__.split(".")[:2])) < (3, 0):
    raise RuntimeError("Install pandas 3.0+ before running the benchmark.")
print("DuckDB:", duckdb.__version__)
print("CPU logical cores:", psutil.cpu_count(logical=True))
print("RAM GiB:", round(psutil.virtual_memory().total / 2**30, 2))


## Part 1: Download dataset

During the ADAC Workshop 1, we will use [Microsoft Security Incident Prediction Dataset](https://www.kaggle.com/datasets/Microsoft/microsoft-security-incident-prediction/data) from Kaggle. First we need to download it from Kaggle, using `kagglehub` library.

In [ ]:
import kagglehub
path = kagglehub.dataset_download("Microsoft/microsoft-security-incident-prediction")
path

In [ ]:
!du -sh {path}

In [ ]:
!find {path} -ls

In [ ]:
import os
train_path = os.path.join(path, "GUIDE_Train.csv")
test_path = os.path.join(path, "GUIDE_Test.csv")
train_path, test_path

### Part 1.2: Exploring the dataset

Next, you need to explore the dataset on [Kaggle incident dataset webpage](https://www.kaggle.com/datasets/Microsoft/microsoft-security-incident-prediction/data).

You can also explore it by performing data exploration analysis in Python below (statistical information about columns, data quantity, making simple visualisations).

In [ ]:
# Data exploration analysis (optional)


### 1.3: Partitioning the dataset

In order to optimize the performance of the queries, we need to partition the dataset. We will use the `ts_date_bucketed` column to partition the dataset.

In [ ]:
import shutil
# --- 1.3 Partitioning the dataset -------------------------------------------------
# Path constants used by the benchmark helpers (Part 2) and by Task 2.5.
WORKDIR = Path("data_adac1"); WORKDIR.mkdir(exist_ok=True)
EVENTS_PATH            = str(WORKDIR / "guide_train.parquet")          # default single-file Parquet
OPTIMIZED_EVENTS_PATH  = str(WORKDIR / "guide_train_sorted.parquet")   # sorted on the filter keys + small row groups
PARTITIONED_EVENTS_DIR = str(WORKDIR / "guide_train_by_date")          # Hive-partitioned by ts_date_bucketed
CSV_EVENTS_PATH        = str(WORKDIR / "guide_q1_slice.csv")           # flat CSV negative baseline (only the query's columns)

# (a) one-off CSV -> single Parquet (the dataset ships as CSV; Task 2 / 2.5 read Parquet)
if not os.path.exists(EVENTS_PATH):
    t0 = time.perf_counter()
    pl.scan_csv(train_path, infer_schema_length=20_000).sink_parquet(EVENTS_PATH, compression="zstd")
    print(f"csv -> parquet in {time.perf_counter() - t0:.1f} s")

con = duckdb.connect(); con.execute("SET enable_progress_bar = false")

# (b) derive ts_date_bucketed: the Timestamp dump is heavily front-tailed -- a few thousand rows are
#     scattered across late-2023 / early-2024, then ~99.5% of the rows fall in ~4 weeks of May-Jun 2024.
#     A raw daily Hive partition would create ~150 near-empty directories, so we keep each "dense" day as
#     its own partition and collapse the oldest ~0.5% of rows into one "_other" partition -> ~30 fairly
#     even partitions. (Rationale, metrics and plots: exploratory-analysis/report.html.)
TAIL_FRACTION = 0.005
date_sql   = 'CAST(substr("Timestamp", 1, 10) AS DATE)'
cutoff     = con.execute(f"SELECT quantile_disc({date_sql}, {TAIL_FRACTION}) FROM read_parquet('{EVENTS_PATH}')").fetchone()[0]
bucket_sql = f"CASE WHEN {date_sql} >= DATE '{cutoff}' THEN CAST({date_sql} AS VARCHAR) ELSE '_other' END"
print(f"ts_date_bucketed: dense-day cutoff = {cutoff} (rows older than this date -> '_other')")

# (c) build the three alternative physical layouts with DuckDB COPY
t0 = time.perf_counter()
shutil.rmtree(PARTITIONED_EVENTS_DIR, ignore_errors=True)
con.execute(f"COPY (SELECT *, {bucket_sql} AS ts_date_bucketed FROM read_parquet('{EVENTS_PATH}')) "
            f"TO '{PARTITIONED_EVENTS_DIR}' (FORMAT PARQUET, COMPRESSION ZSTD, PARTITION_BY (ts_date_bucketed), OVERWRITE_OR_IGNORE TRUE)")
con.execute(f"COPY (SELECT * FROM read_parquet('{EVENTS_PATH}') ORDER BY Category, IncidentGrade, CountryCode) "
            f"TO '{OPTIMIZED_EVENTS_PATH}' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 100000, OVERWRITE_OR_IGNORE TRUE)")
con.execute(f"COPY (SELECT Category, IncidentGrade, CountryCode, IncidentId, MitreTechniques FROM read_parquet('{EVENTS_PATH}')) "
            f"TO '{CSV_EVENTS_PATH}' (FORMAT CSV, HEADER TRUE)")
print(f"built alternative layouts in {time.perf_counter() - t0:.1f} s")

# (d) report sizes + how even the date partitions turned out
pp = con.execute(f"SELECT COUNT(*) n FROM read_parquet('{PARTITIONED_EVENTS_DIR}/**/*.parquet', hive_partitioning=true) GROUP BY ts_date_bucketed").df()["n"]
con.close()
def _dirsize(p): return os.path.getsize(p) if os.path.isfile(p) else sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(p) for f in fs)
def _nfiles(p):  return 1 if os.path.isfile(p) else sum(len(fs) for _, _, fs in os.walk(p))
for name, p in [("default parquet (1 file)", EVENTS_PATH), ("sorted parquet (rg=100k)", OPTIMIZED_EVENTS_PATH),
                ("partitioned parquet (by date bucket)", PARTITIONED_EVENTS_DIR), ("csv slice (5 cols, flat)", CSV_EVENTS_PATH)]:
    print(f"  {name:38s} {_dirsize(p)/2**20:9.1f} MB   {_nfiles(p):3d} file(s)")
print(f"  -> {len(pp)} date partitions; rows/partition  min={pp.min():,}  median={int(pp.median()):,}  max={pp.max():,}")


## Part 2: Measuring performance

You must use one consistent benchmark protocol for all libraries/engines.

Minimum requirements:

1. Run every benchmark at least three times. Five repetitions are recommended.
2. Run `gc.collect()` before each measured repetition to reduce noise from previous Python allocations.
3. Report median runtime, not only one measurement.
4. Record peak memory where possible.
5. Check that results are logically equivalent across libraries/engines.
6. Store your results in a table.
7. Describe any library/engine-specific settings, such as Pandas dtype backend, thread count, Spark local mode, or DuckDB threads.

**Important for memory benchmarks**: notebook kernels keep allocations and library state between cells. Peak-RSS comparisons are often misleading when all variants run in the same process. For Task 4 and any memory-sensitive comparison, prefer running each variant in a fresh process or a small standalone script. If you cannot do that, clearly state this limitation.

You may use the implemented helper below.


In [ ]:
import hashlib, threading
import pyarrow.parquet as pq
# --- Part 2: Measuring performance ------------------------------------------------------------------
# Complete benchmark harness -- you do NOT edit this cell. In Tasks 2-5 you call benchmark_case(...) to
# time a query in one library/engine, and show_results() to get the results table. Path constants
# (EVENTS_PATH / PARTITIONED_EVENTS_DIR / OPTIMIZED_EVENTS_PATH / CSV_EVENTS_PATH) come from Part 1.3.
REPEATS = 5  # >= 3 required, 5 recommended (Part 2 protocol); per-call override via benchmark_case(..., repeats=)

BENCHMARK_COLUMNS = [
    "library_engine", "mode", "query_name", "data_format", "layout",
    "rows", "median_time_s", "iqr_time_s", "peak_memory_mb", "input_size_mb",
    "files", "row_groups", "result_check", "notes",
]

def parquet_source(layout):
    if layout == "unpartitioned": return str(EVENTS_PATH)
    if layout == "partitioned":   return str(PARTITIONED_EVENTS_DIR)
    if layout == "optimized":     return str(OPTIMIZED_EVENTS_PATH)
    raise ValueError(layout)

def parquet_glob(layout):
    if layout == "unpartitioned": return str(EVENTS_PATH)
    if layout == "partitioned":   return os.path.join(str(PARTITIONED_EVENTS_DIR), "**", "*.parquet")
    if layout == "optimized":     return str(OPTIMIZED_EVENTS_PATH)
    raise ValueError(layout)

def layout_profile(layout, data_format="parquet"):
    if data_format == "csv":
        return {"files": 1, "row_groups": None, "input_size_mb": round(os.path.getsize(CSV_EVENTS_PATH) / 2**20, 2)}
    path = Path(parquet_source(layout))
    files = sorted(path.rglob("*.parquet")) if path.is_dir() else [path]
    row_groups = sum(pq.ParquetFile(f).num_row_groups for f in files)
    total_bytes = sum(f.stat().st_size for f in files)
    return {"files": len(files), "row_groups": row_groups, "input_size_mb": round(total_bytes / 2**20, 2)}

def to_pandas_result(result):
    if isinstance(result, pd.DataFrame): return result.copy()
    if isinstance(result, pl.DataFrame): return result.to_pandas()
    return pd.DataFrame(result)

def normalize_checksum_value(value):
    if isinstance(value, np.ndarray):        return json.dumps([normalize_checksum_value(v) for v in value.tolist()], sort_keys=True)
    if isinstance(value, (list, tuple)):     return json.dumps([normalize_checksum_value(v) for v in value], sort_keys=True)
    if pd.isna(value):                       return "<NA>"
    if isinstance(value, (np.bool_, bool)):  return "true" if bool(value) else "false"
    if isinstance(value, (np.integer, int)): return str(int(value))
    if isinstance(value, (np.floating, float)):
        value = float(value)
        return str(int(value)) if value.is_integer() else f"{value:.6f}"
    if isinstance(value, (pd.Timestamp, np.datetime64)): return str(pd.Timestamp(value).isoformat())
    return str(value)

def result_checksum(result):
    pdf = to_pandas_result(result).copy()
    pdf.columns = [str(c) for c in pdf.columns]
    for col in pdf.columns: pdf[col] = pdf[col].map(normalize_checksum_value)
    sort_cols = list(pdf.columns)
    if sort_cols: pdf = pdf.sort_values(sort_cols).reset_index(drop=True)
    return hashlib.md5(pdf.to_csv(index=False, lineterminator="\n").encode("utf-8")).hexdigest()[:12]

benchmark_results = []

def run_with_peak_delta(fn, interval=0.01):
    process = psutil.Process(os.getpid())
    def rss_mb(): return process.memory_info().rss / 2**20
    baseline = rss_mb(); peak = [baseline]; running = [True]
    def sample_memory():
        while running[0]:
            peak[0] = max(peak[0], rss_mb()); time.sleep(interval)
    sampler = threading.Thread(target=sample_memory, daemon=True); sampler.start()
    try:
        result = fn(); peak[0] = max(peak[0], rss_mb())
        return max(peak[0] - baseline, 0.0), result
    finally:
        running[0] = False; sampler.join(timeout=1)

def benchmark_case(library_engine, mode, query_name, layout, fn, repeats=REPEATS, notes="", data_format="parquet"):
    times = []; result = None; peak_mem = None
    profile = layout_profile(layout, data_format=data_format)
    for i in range(repeats):
        gc.collect(); start = time.perf_counter()
        if i == 0: peak_mem, result = run_with_peak_delta(fn)
        else:      result = fn()
        times.append(time.perf_counter() - start)
    pdf = to_pandas_result(result)
    row = {
        "library_engine": library_engine, "mode": mode, "query_name": query_name,
        "data_format": data_format, "layout": layout, "rows": len(pdf),
        "median_time_s": round(float(np.median(times)), 4),
        "iqr_time_s": round(float(np.percentile(times, 75) - np.percentile(times, 25)), 4),
        "peak_memory_mb": round(float(peak_mem), 2),
        "input_size_mb": profile["input_size_mb"], "files": profile["files"], "row_groups": profile["row_groups"],
        "result_check": result_checksum(pdf),
        "notes": (notes + "; " if notes else "") + "RSS delta sampled in notebook process",
    }
    benchmark_results.append(row)
    return row

def show_results():
    return pd.DataFrame(benchmark_results).sort_values(["query_name", "layout", "library_engine", "mode"]).reset_index(drop=True)

print(f"benchmark harness ready -- {len(BENCHMARK_COLUMNS)} result columns; call benchmark_case(...) / show_results()")


## Part 3: Student tasks

### Task 1: Design two benchmark queries

Create two queries of your own choice. They must test different behavior.

Your queries should cover at least two of the following classes:

- selective filter plus aggregation,
- high-cardinality group-by,
- top-k or sorting,
- list/tag explode,
- window or rolling computation,
- query that produces a large output,
- query sensitive to partitioned vs. unpartitioned layout,
- query sensitive to column pruning, predicate pushdown, or row-group pruning.

For each query, write a short hypothesis before you run it:

- what does this query test?
- which library/engine do you expect to perform best?
- which library/engine may use the most memory?
- which physical layout should help, if any?


In [ ]:
# TODO: Define your two query specifications in prose or structured metadata.
# Do not start benchmarking before you can explain what each query is supposed to test.

### Task 2: Benchmark local libraries/engines

Implement your two queries in:

- Pandas 3.0.2 with the default NumPy-backed output from `pd.read_parquet(path)`,
- Pandas 3.0.2 with `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`,
- Polars,
- DuckDB.

For Polars, benchmark at least:

- eager execution,
- lazy execution with default collection,
- lazy execution with streaming engine.


In [ ]:
# TODO: Pandas implementations of your two queries.
# Implement both Pandas read variants:
# 1. default backend: pd.read_parquet(path)
# 2. PyArrow backend: pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")
#
# Report dtypes for both variants and compare runtime/memory.


In [ ]:
# TODO: Polars implementations of your two queries.
# Required modes:
# - eager: read_parquet -> transformations
# - lazy default: scan_parquet -> transformations -> collect()
# - lazy streaming: scan_parquet -> transformations -> collect(engine="streaming")

In [ ]:
# TODO: DuckDB SQL implementations of your two queries.
# Consider querying Parquet files directly instead of first loading all data into Pandas.

### Task 3: File format and Parquet layout optimization

Choose one of your two queries and test whether physical layout changes the amount of data read and the runtime.

Required comparison:

- default Parquet layout: randomly ordered data, one file or the default layout from your generator,
- optimized Parquet layout: choose a layout based on the query pattern, for example sorting by filter columns, changing `row_group_size`, partitioning by a selective column, or using writer-level pruning aids such as bloom filters if your writer and reader clearly support them,
- negative baseline: CSV or JSON/JSONL for the same query, to show what is lost without Parquet column pruning and predicate pushdown.

Use CSV if you do not have a strong reason to prefer JSON/JSONL. If your full dataset contains nested/list columns, create a flat query-specific CSV/JSON baseline containing only the columns needed by the selected query.

Report at least:

- file format and physical layout,
- total input size and number of files,
- runtime and peak memory,
- result checksum/equivalence,
- evidence of pruning where available: query plan, number of files read/skipped, row groups read/skipped, or a clear explanation if the engine does not expose these metrics.

Do not just create a faster layout accidentally. Explain why the layout should help this query.


In [ ]:
# TODO 3: Build and benchmark one optimized layout for one selected query.
# Suggested steps:
# 1. Choose one query with a selective filter or column subset.
# 2. Write a baseline Parquet file/directory.
# 3. Write an optimized Parquet file/directory, e.g. sorted and with a selected row_group_size.
# 4. Write CSV or JSONL as a required negative baseline.
#    If your full dataset has nested/list columns, write a flat query-specific baseline with the columns needed by the selected query.
# 5. Benchmark the same logical query on default Parquet, optimized Parquet, and CSV/JSONL.
# 6. Record IO/pruning evidence where available.

# YOUR CODE HERE


### Task 4: Execution Modes & Analysis

**Goal**: deep dive into execution models, memory limits, and the decision boundary between single-node and distributed processing.

#### Lazy vs. eager vs. streaming

Use Polars to compare execution time and peak memory for the same logical operation in these modes:

- eager execution: `read_parquet` -> filter/transform,
- lazy execution: `scan_parquet` -> filter/transform -> `collect()`,
- streaming execution: `scan_parquet` -> filter/transform -> `collect(engine="streaming")`,
- streaming output: `scan_parquet` -> filter/transform -> `sink_parquet(...)`.

Important distinction:

- `collect(engine="streaming")` uses the streaming engine but still materializes the final result as a DataFrame.
- `sink_parquet(...)` or another sink writes the result to disk and is the better pattern when the output may be large.

Choose a query where this distinction matters. A tiny aggregate result may not show meaningful peak-memory differences. A better stress case keeps many rows, selects several columns, performs a non-trivial filter, and writes a large output.

**Run memory-sensitive variants in separate processes if possible.** If you run all modes in one notebook kernel, previous allocations and engine caches can hide the real memory difference. At minimum, call `gc.collect()` before each measured run and discuss the limitation.

If peak memory is almost identical across modes, increase the dataset size, increase the output size, measure each mode in a fresh process, or explain why your query is not memory-stressful enough.


In [ ]:
# TODO 4: Implement Polars execution-mode experiments.
#
# Required variants:
# 1. eager: read_parquet -> filter/transform
# 2. lazy: scan_parquet -> filter/transform -> collect()
# 3. streaming collect: scan_parquet -> filter/transform -> collect(engine="streaming")
# 4. streaming sink: scan_parquet -> filter/transform -> sink_parquet(...)
#
# Recommended:
# - use a query whose output has many rows, not a tiny aggregate table,
# - measure each mode in a fresh process if possible,
# - call gc.collect() before each measured run,
# - record runtime, peak memory, output row count, and output size,
# - append results to benchmark_results.

# YOUR CODE HERE


### Task 5: Thread and core scalability

Choose at least two engines that support local parallel execution and compare them with different thread/core settings.

Suggested settings:

- DuckDB: configure number of threads for the connection.
- Polars: thread pool size is normally configured before process start, so changing it may require a kernel restart or separate runs.

In your report, do not only show speedup. Explain why scaling is or is not close to linear.

Hints:
- Polars' thread pool is built from `POLARS_MAX_THREADS` at `import polars` time; there is no runtime setter, so each thread-count must run in a fresh process/kernel.
- DuckDB threads, by contrast, can be changed on a live connection with `SET threads=N`.

In [ ]:
# TODO: Run selected scalability experiments and append results to benchmark_results.

## Final notebook report

The rendered notebook is your final submission. You do not submit a separate report.

Before submitting, make sure this notebook contains:

- full name,
- link to this notebook in your fork,
- two query descriptions with hypotheses,
- local benchmark table for Pandas 3.0.2 default backend, Pandas 3.0.2 PyArrow backend, Polars, DuckDB,
- file-format and Parquet-layout experiment with a required CSV/JSON negative baseline and evidence about column pruning, predicate pushdown, file pruning, or row-group pruning,
- Polars eager vs. lazy vs. streaming vs. sink discussion,
- local scalability results for selected libraries/engines,
- plots or tables that support your claims,
- final recommendations.

Do not commit generated data files, benchmark outputs, credentials, or local environment files.


### Final answers

Fill in the cells below. These answers should be visible in the rendered notebook.

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 1: Which query best exposes the difference between DataFrame and SQL engines?
FINAL_ANSWER_1 = """
TODO: Write your answer here.
"""
display_answer("Final answer 1", FINAL_ANSWER_1)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 2: Which query is most memory-sensitive?
FINAL_ANSWER_2 = """
TODO: Write your answer here. Refer to measured peak memory and dataset/query shape.
"""
display_answer("Final answer 2", FINAL_ANSWER_2)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 3: Did lazy execution change the amount of data read or materialized?
FINAL_ANSWER_3 = """
TODO: Write your answer here. Refer to predicate/projection pushdown or query plans if available.
"""
display_answer("Final answer 3", FINAL_ANSWER_3)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 4: Did streaming collection reduce memory, runtime, or both?
FINAL_ANSWER_4 = """
TODO: Write your answer here. Distinguish collect(engine="streaming") from sink_parquet(...).
"""
display_answer("Final answer 4", FINAL_ANSWER_4)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 5: When was a streaming sink more appropriate than collecting the result?
FINAL_ANSWER_5 = """
TODO: Write your answer here. Mention output size and whether the final result needed to be materialized in Python.
"""
display_answer("Final answer 5", FINAL_ANSWER_5)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 6: How did Pandas default backend compare with the PyArrow dtype backend?
FINAL_ANSWER_6 = """
TODO: Write your answer here. Mention runtime, memory, dtypes, and whether string-heavy or IO-heavy queries changed the result.
"""
display_answer("Final answer 6", FINAL_ANSWER_6)
